# 📌 NLP Assignment - 5  
## Fine-Tuning BERT for POS Tagging & Chunking

---

## 🎯 Key Objectives

- Implement token classification using transformer models  
- Perform chunking using CoNLL-2003 dataset  
- Apply tokenization and label alignment  
- Train a DistilBERT model efficiently  
- Evaluate model performance  
- Perform inference on custom sentences  


In [1]:
# ===============================
# 🔧 Setup & Warning Suppression
# ===============================

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import os
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from seqeval.metrics import classification_report, f1_score, accuracy_score
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 📂 Data Loading

In [2]:
def read_conll(file_path):
    sentences = []
    labels = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        words, tags = [], []
        
        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words, tags = [], []
            else:
                parts = line.split()
                words.append(parts[0])
                tags.append(parts[-1])
        
        if words:
            sentences.append(words)
            labels.append(tags)
    
    return sentences, labels


print("Loading dataset...")

train_sentences, train_labels = read_conll("conll2003/eng.train")
val_sentences, val_labels = read_conll("conll2003/eng.testa")

print("✅ Dataset Loaded!")
print("Training samples:", len(train_sentences))

Loading dataset...
✅ Dataset Loaded!
Training samples: 14987


## 🔍 Dataset Overview

In [3]:
print("Sample sentence:", train_sentences[0])
print("Sample labels:", train_labels[0])

Sample sentence: ['-DOCSTART-']
Sample labels: ['O']


## 🧹 Data Preprocessing

In [4]:
train_sentences = train_sentences[:2000]
train_labels = train_labels[:2000]

val_sentences = val_sentences[:500]
val_labels = val_labels[:500]

print("Reduced dataset size ✅")

Reduced dataset size ✅


## 🔖 Label Mapping

In [5]:
unique_labels = set(label for sent in train_labels for label in sent)

label_list = list(unique_labels)

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print("Labels:", label_list)

Labels: ['B-ORG', 'I-PER', 'I-LOC', 'I-MISC', 'B-PER', 'O', 'B-LOC', 'I-ORG', 'B-MISC']


## 🔄 Tokenization

In [6]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align(sentences, labels):

    tokenized_inputs = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        is_split_into_words=True
    )

    aligned_labels = []

    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []

        previous_word_idx = None

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:

                if word_idx < len(label):
                    label_ids.append(label2id[label[word_idx]])
                else:
                    label_ids.append(-100)

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs


train_encodings = tokenize_and_align(train_sentences, train_labels)
val_encodings = tokenize_and_align(val_sentences, val_labels)

print("Tokenization done ✅")

Tokenization done ✅


## 📦 Dataset Preparation

In [7]:
print("Creating dataset objects...")

class NERDataset(torch.utils.data.Dataset):

    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])


# Create datasets
train_dataset = NERDataset(train_encodings)
val_dataset = NERDataset(val_encodings)

print("Datasets ready ✅")

Creating dataset objects...
Datasets ready ✅


## 🏗 Model Setup

In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print("Model ready ✅")

Model ready ✅


## ⚙️ Data Collator

In [9]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Data collator ready ✅")

Data collator ready ✅


## 📊 Evaluation Metrics

In [10]:
def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [id2label[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {
        "accuracy": accuracy_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions)
    }

## ⚙️ Training

In [11]:
print("Initializing training...")

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=16,
    num_train_epochs=3,              
    weight_decay=0.01,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready ✅")

Initializing training...
Trainer ready ✅


In [12]:
trainer.train()

{'loss': 1.6206, 'grad_norm': 6.495283126831055, 'learning_rate': 1.9466666666666668e-05, 'epoch': 0.08}
{'loss': 0.8144, 'grad_norm': 1.3801343441009521, 'learning_rate': 1.8933333333333334e-05, 'epoch': 0.16}
{'loss': 0.6977, 'grad_norm': 1.354827880859375, 'learning_rate': 1.8400000000000003e-05, 'epoch': 0.24}
{'loss': 0.6006, 'grad_norm': 1.3889166116714478, 'learning_rate': 1.7866666666666666e-05, 'epoch': 0.32}
{'loss': 0.4259, 'grad_norm': 1.2946726083755493, 'learning_rate': 1.7333333333333336e-05, 'epoch': 0.4}
{'loss': 0.3657, 'grad_norm': 1.269527792930603, 'learning_rate': 1.6800000000000002e-05, 'epoch': 0.48}
{'loss': 0.3892, 'grad_norm': 2.1032180786132812, 'learning_rate': 1.6266666666666668e-05, 'epoch': 0.56}
{'loss': 0.33, 'grad_norm': 1.2880219221115112, 'learning_rate': 1.5733333333333334e-05, 'epoch': 0.64}
{'loss': 0.3504, 'grad_norm': 1.43452787399292, 'learning_rate': 1.5200000000000002e-05, 'epoch': 0.72}
{'loss': 0.244, 'grad_norm': 1.1446062326431274, 'lear

TrainOutput(global_step=375, training_loss=0.24895044803619384, metrics={'train_runtime': 2502.9711, 'train_samples_per_second': 2.397, 'train_steps_per_second': 0.15, 'train_loss': 0.24895044803619384, 'epoch': 3.0})

## 📊 Evaluation Results

In [13]:
results = trainer.evaluate()
results

{'eval_loss': 0.1157667264342308, 'eval_accuracy': 0.9679326406305984, 'eval_f1': 0.8301449275362318, 'eval_runtime': 50.6555, 'eval_samples_per_second': 9.871, 'eval_steps_per_second': 0.632, 'epoch': 3.0}


{'eval_loss': 0.1157667264342308,
 'eval_accuracy': 0.9679326406305984,
 'eval_f1': 0.8301449275362318,
 'eval_runtime': 50.6555,
 'eval_samples_per_second': 9.871,
 'eval_steps_per_second': 0.632,
 'epoch': 3.0}

## 🔮 Inference

In [14]:
sentence = ["John", "works", "at", "Google"]

tokens = tokenizer(sentence, return_tensors="pt", is_split_into_words=True)

outputs = model(**tokens).logits

predictions = np.argmax(outputs.detach().numpy(), axis=2)

predicted_labels = [id2label[p] for p in predictions[0]]

print(list(zip(sentence, predicted_labels[:len(sentence)])))

[('John', 'O'), ('works', 'B-PER'), ('at', 'O'), ('Google', 'O')]


## 📈 Observations

- The DistilBERT model performs efficiently with reduced dataset size  
- Tokenization and label alignment significantly impact performance  
- Chunking tasks require careful handling of subword tokens  
- Pretrained models reduce training time and improve accuracy  

---

## 🧾 Conclusion

- This project successfully demonstrates the implementation of token classification using transformer models.

- Using the CoNLL-2003 dataset, we performed chunking efficiently with reduced     training time.

- The model achieved reliable performance, highlighting the effectiveness of transformer-based approaches in NLP tasks.

---